In [1]:
print("Hello World")

Hello World


In [36]:
import numpy as np 
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [10]:
df = pd.read_csv("Titanic-cleaned.csv")

In [4]:
df.isnull().sum()

Unnamed: 0     0
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
Has_Cabin      0
dtype: int64

In [5]:
df.head(10)

,Unnamed: 0,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,Has_Cabin
0,0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,0
1,1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,1
2,2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,0
3,3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,1
4,4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,0
5,5,6,0,3,"Moran, Mr. James",male,24.0,0,0,330877,8.4583,Q,0
6,6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,S,1
7,7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,S,0
8,8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,S,0
9,9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,C,0


In [3]:
y = df['Survived']
X = df.drop('Survived', axis=1)

In [4]:
X = X.drop(['Name', 'Ticket', 'Cabin'], axis=1, errors='ignore')
X['Sex'] = X['Sex'].map({'male': 0, 'female': 1})
X = pd.get_dummies(X, columns=['Embarked'], drop_first=True)

In [7]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  

In [ ]:
X['Leak_Feature']=y

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

In [9]:
model = LogisticRegression()
model.fit(X_train, y_train)

preds = model.predict(X_test)
print("Leaked Accuracy:", accuracy_score(y_test, preds))

Leaked Accuracy: 1.0


In [37]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   
X_test_scaled = scaler.transform(X_test)         

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([('scaler', StandardScaler()),('model', LogisticRegression())])

pipeline.fit(X_train, y_train)
preds = pipeline.predict(X_test)

print("Pipeline Accuracy:", accuracy_score(y_test, preds))

Pipeline Accuracy: 0.8435754189944135


In [12]:
df_model = df.copy()
df_model['Sex_num'] = (df_model['Sex'] == 'female').astype(int)
df_model['Embarked'] = df_model['Embarked'].fillna('S')

In [13]:
encoder = LabelEncoder()
df_model['Embarked_num'] = encoder.fit_transform(df_model['Embarked'])


In [15]:
features = ['Pclass','Age','SibSp','Parch','Fare','Has_Cabin','Sex_num','Embarked_num']

X = df_model[features]
X = X.fillna(X.mean())
y = df_model['Survived']


In [ ]:
np.random.seed(42)

df_model['Rescue_Priority'] = (df_model['Survived'] * 0.9 + np.random.normal(0, 0.05, len(df_model)))

print(df_model[['Survived', 'Rescue_Priority']].head(10))

   Survived  Rescue_Priority
0         0         0.024836
1         1         0.893087
2         1         0.932384
3         1         0.976151
4         0        -0.011708
5         0        -0.011707
6         0         0.078961
7         0         0.038372


In [18]:
model = LogisticRegression(max_iter=500)

In [21]:
X_leaked = X.copy()
X_leaked['Rescue_Priority'] = df_model['Rescue_Priority']

score_leaked = cross_val_score(model, X_leaked, y,cv=5,scoring='accuracy')
score_leaked = score_leaked.mean()
print(score_leaked)

1.0


In [23]:
score_clean = cross_val_score(model, X, y,cv=5,scoring='accuracy')
score_clean = score_clean.mean()
print(score_clean)

0.7968614650681063


In [25]:
#ID
X = df_model[['PassengerId', 'Pclass', 'Fare', 'Age']]
X = X.fillna(X.mean())

tree = DecisionTreeClassifier(random_state=42)


In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tree.fit(X_train, y_train)

train_acc = accuracy_score(y_train, tree.predict(X_train))
test_acc  = accuracy_score(y_test, tree.predict(X_test))
print(train_acc)
print(test_acc)

1.0
0.664804469273743


In [29]:
X = df_model[['Pclass', 'Fare', 'Age']]
X = X.fillna(X.mean())

tree = DecisionTreeClassifier(random_state=42)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tree.fit(X_train, y_train)

train_acc = accuracy_score(y_train, tree.predict(X_train))
test_acc  = accuracy_score(y_test, tree.predict(X_test))
print(train_acc)
print(test_acc)

0.952247191011236
0.6703910614525139


In [31]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [32]:
model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)

accuracy = accuracy_score(y_test, model.predict(X_test))
print(accuracy)

0.7486033519553073


In [33]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)   
X_test  = scaler.transform(X_test)       

model.fit(X_train, y_train)

accuracy = accuracy_score(y_test, model.predict(X_test))
print(accuracy)


0.7486033519553073


In [34]:
total_rows = len(df)
duplicate_rows = df.duplicated().sum()

print("Duplicate Check:")
print("Total rows     :", total_rows)
print("Duplicate rows :", duplicate_rows)

Duplicate Check:
Total rows     : 891
Duplicate rows : 0


In [35]:
near_duplicates = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']

near_duplicates = df.duplicated(subset=near_duplicates).sum()
print("Near-duplicates :", near_duplicates)


Near-duplicates : 135


In [37]:
#FIXING:

X = df[['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Has_Cabin', 'Sex', 'Embarked']]
y = df['Survived']


In [38]:
num_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Has_Cabin']
cat_cols = ['Sex', 'Embarked']

In [39]:
num_pipeline = Pipeline([('fill', SimpleImputer(strategy='median')),('scale', StandardScaler())])
cat_pipeline = Pipeline([('fill', SimpleImputer(strategy='most_frequent')),('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])


In [40]:
preprocess = ColumnTransformer([('num', num_pipeline, num_cols),('cat', cat_pipeline, cat_cols)])

In [41]:
model = Pipeline([('preprocess', preprocess),('classifier', LogisticRegression(max_iter=500))])

In [ ]:
scores = cross_val_score(model, X, y, cv=5,scoring='accuracy')

for i, s in enumerate(scores, 1):
    print("Fold", i, ":", round(s * 100, 1), "%")

print("\nAverage Accuracy :", round(scores.mean() * 100, 1), "%")
print("Variation :", round(scores.std() * 100, 1), "%")

Fold 1 : 79.9 %
Fold 2 : 78.7 %
Fold 3 : 79.8 %
Fold 4 : 78.1 %
Fold 5 : 82.0 %

Average Accuracy : 79.7 %
Variation        : 1.4 %
